# Assignment 2 Project

This notebook is a clean wrapper around the reusable code in `src/assignment_2_project`.
It keeps the original notebook untouched and lets you run the workflow from a smaller, easier-to-read notebook.

## Setup

Set your environment variables in `.env` and make sure the project dependencies are installed before running the cells below.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from src.config import get_client, settings
from src.data import build_chunks, build_doc_info, get_embeddings, get_pdf_files, load_financebench_data, load_or_build_vectorstore, sample_task1_rows
from src.evaluation import get_avg_metrics, judge_correctness, run_faithfulness_scores
from src.evaluation import page_hit_at_k
from src.experiments import build_cycle_summary, run_naive_generation, run_rag_answers
from src.rag import SYSTEM_PROMPT, SYSTEM_PROMPT_STRICT

c:\Users\guyil\OneDrive\Desktop\Programing\Courses\nebius\AI_Performance_Engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data and Build Retrieval

This mirrors the notebook workflow, but the implementation lives in reusable modules.

In [ ]:
client = get_client()

df = load_financebench_data(PROJECT_ROOT / 'financebench_merged.jsonl')
doc_info = build_doc_info(df)
pdf_files = get_pdf_files(PROJECT_ROOT / 'pdfs')
embeddings = get_embeddings()
chunks = build_chunks(doc_info, pdf_files)
vectorstore = load_or_build_vectorstore(embeddings, index_dir=PROJECT_ROOT / 'faiss_financebench', chunks=chunks)

print(f'Rows: {len(df)}')
print(f'Documents: {len(doc_info)}')
print(f'Chunks: {len(chunks)}')

## Task 1: Naive Generation

This uses the reusable baseline generation helper and keeps the output in a dataframe.

In [ ]:
task1_df = run_naive_generation(df, client, model=settings.model, limit=10)
task1_df.head()

## Task 5: RAG Answers

Generate answers with retrieval and inspect the retrieved context.

In [ ]:
rag_df = run_rag_answers(
    df=sample_task1_rows(df),
    client=client,
    vectorstore=vectorstore,
    model=settings.model,
    system_prompt=SYSTEM_PROMPT,
    k=4,
)
rag_df.head()

## Evaluation

Use the evaluation helpers to score correctness, page hit, and faithfulness.

In [ ]:
rag_eval_df = rag_df.copy()
rag_eval_df["correctness_raw"] = rag_eval_df.apply(
    lambda row: judge_correctness(row['question'], row['ground_truth'], row['RAG_answer'], client, settings.judge_model),
    axis=1,
)
rag_eval_df["correctness"] = rag_eval_df["correctness_raw"].apply(lambda value: value['verdict'])
rag_eval_df["correctness_justification"] = rag_eval_df["correctness_raw"].apply(lambda value: value['justification'])
for k in [1, 3, 5]:
    rag_eval_df[f'page_hit_at_{k}'] = rag_eval_df.apply(lambda row, kk=k: page_hit_at_k(row, kk), axis=1)
rag_eval_df["faithfulness"] = run_faithfulness_scores(rag_eval_df, limit=5)
get_avg_metrics(rag_eval_df)

## Improvement Cycles

The reusable package also exposes the pieces needed to compare prompt, model, and top-k variants. Compose those helpers here if you want to extend the notebook with the full cycle table.

In [ ]:
print('Notebook wrapper loaded successfully.')